# Level 3 — Asset Management Workflow

**Audience:** analysts who need a reproducible research workflow rather than a
single metric or optimizer call.

**Prerequisites:** Levels 1 and 2.

**Learning goals**

1. separate estimation and evaluation windows;
2. compare optimized portfolios with a transparent benchmark;
3. produce benchmark-relative risk/return diagnostics;
4. document assumptions and limitations.

Legacy labs 119 and 121–129 progress into CPPI, simulation, liabilities,
interest rates, and dynamic allocation. Those modules are a planned extension;
this executable notebook intentionally uses only the toolkit's current public
API.

## 1. Setup and reproducible sample

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

pd.options.display.float_format = "{:.4f}".format

from asset_management_toolkit.analytics import risk_return_stats
from asset_management_toolkit.portfolio import (
    global_minimum_variance,
    maximum_sharpe_ratio,
)

In [ ]:
rng = np.random.default_rng(7)
asset_names = ["Income", "Balanced", "Growth"]
monthly_mean = np.array([0.0035, 0.0055, 0.0075])
monthly_cov = np.array(
    [
        [0.00020, 0.00008, 0.00006],
        [0.00008, 0.00055, 0.00022],
        [0.00006, 0.00022, 0.00120],
    ]
)
all_returns = pd.DataFrame(
    rng.multivariate_normal(monthly_mean, monthly_cov, size=180),
    columns=asset_names,
    index=pd.period_range("2011-01", periods=180, freq="M").to_timestamp("M"),
)
estimation = all_returns.iloc[:120]
evaluation = all_returns.iloc[120:]

## 2. Estimate weights without looking at the evaluation period

All annualized inputs below come only from the first 120 months.

In [ ]:
expected = estimation.mean() * 12
covariance = estimation.cov() * 12

weights = pd.DataFrame(
    {
        "GMV": global_minimum_variance(covariance),
        "Max Sharpe": maximum_sharpe_ratio(
            0.02, expected, covariance
        ),
        "Equal weight": pd.Series(1 / len(asset_names), index=asset_names),
    }
)
weights

## 3. Evaluate realized monthly returns

The equal-weight portfolio is the benchmark. This is an explicit comparison
choice, not a claim that it is the investable market portfolio.

In [ ]:
portfolio_returns = pd.DataFrame(
    {
        name: evaluation.mul(weight, axis=1).sum(axis=1)
        for name, weight in weights.items()
    }
)
benchmark = portfolio_returns["Equal weight"].rename("Benchmark")
portfolio_returns.head()

In [ ]:
review = risk_return_stats(
    portfolio_returns[["GMV", "Max Sharpe"]],
    benchmark=benchmark,
    risk_free_rate=0.02,
    periods_per_year=12,
)
review.T

## 4. A compact decision record

Record method, observation window, benchmark, and limitations alongside the
numbers. This prevents a metric table from becoming detached from its
assumptions.

In [ ]:
decision_record = {
    "estimation_window": (
        str(estimation.index.min().date()),
        str(estimation.index.max().date()),
    ),
    "evaluation_window": (
        str(evaluation.index.min().date()),
        str(evaluation.index.max().date()),
    ),
    "frequency": "monthly",
    "periods_per_year": 12,
    "benchmark": "equal-weight portfolio over the same three assets",
    "constraints": "long-only, fully invested",
    "limitation": (
        "Synthetic sample; ignores costs, turnover, taxes, and capacity."
    ),
}
decision_record

## Exercise

Re-estimate the portfolios using only the first 60 months, then evaluate on the
same final 60-month window. How sensitive are the weights and realized
statistics to the estimation sample?

In [ ]:
short_estimation = all_returns.iloc[:60]
short_expected = short_estimation.mean() * 12
short_covariance = short_estimation.cov() * 12

# Build short_weights, realized portfolio returns, and a new summary here.

### Answer scaffold

Use the same sequence as Sections 2–3. Compare both weights and realized
metrics; do not judge robustness from Sharpe ratio alone.

In [ ]:
short_weights = pd.DataFrame(
    {
        "GMV": global_minimum_variance(short_covariance),
        "Max Sharpe": maximum_sharpe_ratio(
            0.02, short_expected, short_covariance
        ),
    }
)
weight_change = short_weights - weights[["GMV", "Max Sharpe"]]
weight_change

## Roadmap from the legacy advanced labs

The following topics are deliberately not presented as working APIs yet:

- Level 3A: CPPI and drawdown constraints (lab 119);
- Level 3B: GBM and Monte Carlo diagnostics (labs 121–123);
- Level 3C: present value, CIR rates, duration, and bonds (labs 124–127);
- Level 3D: fixed mix, glide paths, and dynamic risk budgeting (labs 128–129).

Each topic should enter the library only after provenance review, a small
public API, deterministic tests, and its own executable tutorial.